# Unity Catalog setup Notes and practicals

# Check weather the workspace is attached to Unity Catalog Metastore

In [0]:
%sql
select current_metastore();

# Unity Catalog setup notes
--> In order to use the cloud storage with our DB_WS we need to create unity catalog objects like "Storage Credential" and "Exter Location"

--> Stogare credential is simply a authentication and authorisation mechanism for accessing the data stored in the Azure storage on behalf of the users.

--> We can use managed identity or service principle to connect the ADLS to DB_WS via "storage credential"
to make it more simple we have something called "Access Connector for databricks" in azure, Which lets us to connect a managed identity to a Databricks account. so we will be using this to connect our DB_WS to our ADLS containers.

--> External Location simply combines the Ctorage Credential with colud storage container to gain access to the specific contatiner in the cloud storage.
we may create as many External Locations as we need for the catalogs also we can create the subfolers within the containers the external location refers to and assign them with each of the catalogs or schemas, this way the data for each of the catalogs or schemas can be kept separate and organised. 

--> As we discussed we can create the stogare credential using a managed identiy or service principle. 
Once we create the access connector we need to assign the role "Storage Blog Data Contributor" to the Access Connter to the datalake so that the access connector can access the Datalake. we will then create the storage connector using the access connector information with that the storage credential will also be able to access the datalake via the access connector.we then need to create the "external location" .

--> As we said external location is an object that combines the storage credential and the ADLS container, when an user refernces to the external location Unity Catalog knows which storage cred to use for access and if the storage cred has access to the storage acc the authentication succeeds. You can apply access controls to both stroge cred and Ext Loc, this lets the admins to manages the access to the storage at the more granular level. if the users does not have access to the storage cred or ext log the Unity Catalog fails to authenticate.



![image_1780557702378.png](./image_1780557702378.png "image_1780557702378.png")

# Steps to carry out to implement the configuration of access to cloud storage

1. Create Access Connector 
2. Create a ADLS 
3. Assign Storage Blob Data Contributor role
4. Create Storage Credential
5. Create External Location

--> Storage credential is a Databricks Unity Catalog object which is a credential but underneeth it is a "access connector", underneath it is a "managed identity".

so Storage Credential == Access Connector == Managed Identity

--> we can create the Storage Credential from the databricks UI as mentioned below
1. go to catalog explorer

2. click on create on right of the catalog explorer screen

3. then click on create credential
and give necessary details like credential name and access connector id
we can get the access connector id from the azure portal and go to the resource and then copy the resource id
and then click on create.

Now that we created the "Storage Credential" we then go ahead and create the "External Location". 
we can do that from the UI as well but better to create it via SQL scripts (as shown below) so that we can repeate the process in any environment.


## Access the cloud storage without creating the External Location.
example: %fs ls 'abfss://<container_name>@<storageacc_name>.dfs.core.windows.net'

In [0]:
%fs ls 'abfss://landing@databricksdemo001.dfs.core.windows.net'

## Access the cloud storage after creating the External Location.
example: %fs ls 'abfss://<container_name>@<storageacc_name>.dfs.core.windows.net'

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS databricksdemo001_landing
URL 'abfss://landing@databricksdemo001.dfs.core.windows.net'
WITH (STORAGE CREDENTIAL dea_course_ext_sc)
COMMENT 'Demo for Storage Credential and External Location creation with notes'

In [0]:
%fs ls 'abfss://landing@databricksdemo001.dfs.core.windows.net'

### since we just created the External Location and there are not files in it it simply shows OK rather than printing the contents in the storage container